# Model Training
This notebook serves to run model training runs using scikit-learn's random forest classifier model, with MLflow for metric tracking.

In [0]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.impute import SimpleImputer
import mlflow
import mlflow.sklearn

In [0]:
# load train and test data from notebook 03
train_path = '/Volumes/workspace/default/microbiome_project_files/processed/train_data.parquet'
test_path = '/Volumes/workspace/default/microbiome_project_files/processed/test_data.parquet'

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

print(f"train: {train_df.shape}")
print(f"test: {test_df.shape}")

train: (345, 1874)
test: (87, 1874)


In [0]:
# separate features and target
X_train = train_df.drop(['sample_name', 'empo_3'], axis=1)
y_train = train_df['empo_3']

X_test = test_df.drop(['sample_name', 'empo_3'], axis=1)
y_test = test_df['empo_3']

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

X_train: (345, 1872)
X_test: (87, 1872)


In [0]:
# replace 'not applicable' with np.nan
X_train = X_train.replace('not applicable', np.nan)
X_test = X_test.replace('not applicable', np.nan)

# impute missing values with mean
imputer = SimpleImputer(
    strategy='mean'
)
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [0]:
# baseline random forest with class_weight='balanced'
mlflow.autolog()

with mlflow.start_run(run_name="baseline_rf"):
    rf = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced',  # handles class imbalance
        n_jobs=-1
    )
    
    rf.fit(X_train, y_train)
    
    # predictions
    y_pred = rf.predict(X_test)
    
    # metrics
    accuracy = accuracy_score(y_test, y_pred)
    print(f"accuracy: {accuracy:.3f}")
    print(f"\nclassification report:\n{classification_report(y_test, y_pred)}")
    
    mlflow.log_metric("test_accuracy", accuracy)

2025/12/28 20:27:18 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/12/28 20:27:18 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'


accuracy: 0.609

classification report:
                         precision    recall  f1-score   support

          Animal corpus       0.67      0.57      0.62         7
      Animal distal gut       0.50      0.71      0.59        14
    Animal proximal gut       0.50      0.20      0.29         5
       Animal secretion       0.00      0.00      0.00         4
          Fungus corpus       1.00      0.50      0.67         2
          Plant surface       0.60      0.60      0.60        10
  Sediment (non-saline)       0.88      1.00      0.93         7
      Sediment (saline)       0.75      0.69      0.72        13
      Soil (non-saline)       0.46      0.75      0.57         8
Subsurface (non-saline)       0.00      0.00      0.00         5
       Surface (saline)       0.00      0.00      0.00         1
     Water (non-saline)       1.00      0.50      0.67         4
         Water (saline)       0.88      1.00      0.93         7

               accuracy                         

/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [0]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("confusion matrix:")
print(cm)

confusion matrix:
[[ 4  1  0  1  0  0  0  0  1  0  0  0  0]
 [ 2 10  0  0  0  0  0  0  1  1  0  0  0]
 [ 0  2  1  0  0  1  0  1  0  0  0  0  0]
 [ 0  0  1  0  0  1  0  0  0  1  0  0  1]
 [ 0  0  0  1  1  0  0  0  0  0  0  0  0]
 [ 0  4  0  0  0  6  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  7  0  0  0  0  0  0]
 [ 0  1  0  0  0  2  0  9  0  1  0  0  0]
 [ 0  0  0  0  0  0  1  1  6  0  0  0  0]
 [ 0  1  0  0  0  0  0  0  4  0  0  0  0]
 [ 0  0  0  0  0  0  0  1  0  0  0  0  0]
 [ 0  1  0  0  0  0  0  0  1  0  0  2  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  7]]


The recall of this baseline model is 61%, which is not bad for a dataset with 13 classes and 87 test samples, but tuning may help improve the model.

### Tuning

In [0]:
from sklearn.model_selection import RandomizedSearchCV

n_estimators --> how many decision trees are in the forest

max_depth --> how deep each tree can grow

min_samples_split --> min # of samples needed to split a node

min_samples_leaf --> min # samples required at leaf node

In [0]:
param_dist = {
    'n_estimators': [200, 300, 500], 
    'max_depth': [20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_tuned = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='f1_weighted',
    random_state=42,
    n_jobs=-1
)

In [0]:
rf_tuned = rf_tuned.fit(X_train, y_train)
print(f"best params: {rf_tuned.best_params_}")
print(f"best score: {rf_tuned.best_score_:.3f}")

2025/12/28 20:27:25 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '38728613e09342dfab7a921f6056b6a5', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2025/12/28 20:27:25 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
/databricks/python/lib/python3.12/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
2025/12/28 20:28:08 INFO mlflow.sklearn.utils: Logging the 5 best runs, 5 runs will be omitted.


best params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 30}
best score: 0.640


In [0]:
# evaluate tuned model on test set
with mlflow.start_run(run_name="tuned_rf"):
    best_rf = rf_tuned.best_estimator_
    
    y_pred_tuned = best_rf.predict(X_test)
    
    accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
    print(f"tuned model accuracy: {accuracy_tuned:.3f}")
    print(f"\nclassification report:\n{classification_report(y_test, y_pred_tuned)}")
    
    mlflow.log_metric("test_accuracy", accuracy_tuned)
    mlflow.sklearn.log_model(best_rf, "model")

tuned model accuracy: 0.644

classification report:
                         precision    recall  f1-score   support

          Animal corpus       0.50      0.57      0.53         7
      Animal distal gut       0.53      0.64      0.58        14
    Animal proximal gut       0.33      0.20      0.25         5
       Animal secretion       0.33      0.25      0.29         4
          Fungus corpus       1.00      0.50      0.67         2
          Plant surface       0.78      0.70      0.74        10
  Sediment (non-saline)       0.88      1.00      0.93         7
      Sediment (saline)       0.90      0.69      0.78        13
      Soil (non-saline)       0.50      0.75      0.60         8
Subsurface (non-saline)       0.33      0.40      0.36         5
       Surface (saline)       0.00      0.00      0.00         1
     Water (non-saline)       1.00      0.50      0.67         4
         Water (saline)       0.88      1.00      0.93         7

               accuracy             

/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
2025/12/28 20:28:12 WARNI

In [0]:
cm_tuned = confusion_matrix(y_test, y_pred_tuned)
print("confusion matrix:")
print(cm_tuned)

confusion matrix:
[[4 1 0 1 0 0 0 0 1 0 0 0 0]
 [3 9 0 0 0 0 0 0 1 1 0 0 0]
 [0 2 1 0 0 1 0 0 0 1 0 0 0]
 [0 0 1 1 0 0 0 0 0 1 0 0 1]
 [0 0 0 1 1 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 7 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 7 0 0 0 0 0 0]
 [1 0 0 0 0 1 0 9 1 1 0 0 0]
 [0 0 1 0 0 0 1 0 6 0 0 0 0]
 [0 1 0 0 0 0 0 0 2 2 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 1 0 0 2 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 7]]


The tuned model achieved 64.4% accuracy (up from 61% baseline), with weighted F1 improving from 0.59 to 0.64. Sediment environments showed strong classification performance (88-90% precision), while plant surface prediction improved from 60% to 70% recall. Small classes like subsurface (non-saline) and animal secretion showed modest gains but remain challenging due to limited training samples. The model struggles most with rare classes like surface (saline) with only 1 test sample and biologically similar categories like animal proximal vs. distal gut, which is expected given their overlapping microbial communities.

In [0]:
# save model and results for notebook 05
import pickle

# save the tuned model
model_path = '/Volumes/workspace/default/microbiome_project_files/processed/best_rf_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(best_rf, f)

# save predictions and cv results
results = {
    'y_pred_tuned': y_pred_tuned,
    'cm_tuned': cm_tuned,
    'accuracy_tuned': accuracy_tuned,
    'best_params': rf_tuned.best_params_
}

results_path = '/Volumes/workspace/default/microbiome_project_files/processed/model_results.pkl'
with open(results_path, 'wb') as f:
    pickle.dump(results, f)

rf_tuned_path = '/Volumes/workspace/default/microbiome_project_files/processed/rf_tuned.pkl'
with open(rf_tuned_path, 'wb') as f:
    pickle.dump(rf_tuned, f)

print(f"saved model to {model_path}")
print(f"saved results to {results_path}")

saved model to /Volumes/workspace/default/microbiome_project_files/processed/best_rf_model.pkl
saved results to /Volumes/workspace/default/microbiome_project_files/processed/model_results.pkl
